In [ ]:
## Import modules
import os,sys
import numpy as np
import cftime 
import glob
import json 
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numba 
import concurrent.futures
# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger 
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# Import utilities for this comparison
sys.path.insert(0,cmct_dir)
from cmct.time_utils import check_datarange
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.json_to_netcdf import *

from cmct.gravimetry import *
from cmct.projection import *


In [ ]:
# Ice sheet
loc = 'GIS' # 'GIS' or 'AIS'

# Set time range for comparison
start_year = 2006
end_year = 2010

# Set the observation data dir path
obs_filename = cmct_dir + '/data/calving/observed_icemask_ismip_annual.nc'


# Set the Model Data dir path
if loc == "GIS":
    # Greenland
    mod_filename_template = cmct_dir + '/test/calving/gris*.nc'
elif loc == "AIS":    
    # Antartica
    mod_filename_template = cmct_dir + '/test/calving/ais*.nc'
    
# Output filetype and filename
filetype = 'netcdf' # netcdf or json or None
filename = 'calving_comparison'

In [ ]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# # Check if model file exist
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}")


# Loading Observation Data 


In [ ]:
print(obs_filename)
gsfc = load_gsfc_calving(obs_filename)

# Comparing Ensemble Data

In [ ]:
nc_filenames = glob.glob(mod_filename_template)
if not nc_filenames:
    raise FileNotFoundError(f"No model files found matching template: {mod_filename_template}")
else:
    print(f"Found {len(nc_filenames)} model files matching template: {mod_filename_template}")

# Chunking Process and assign to workers






In [ ]:

# Loop through each file 
for nc_filename in nc_filenames:
    
    
    print(f"\nProcessing:{nc_filename}")

    # Load model data
    model_res = load_model_calving(nc_filename)
    time_var = model_res['time']
    
    # Convert start/end comparison times to fractional year
    calendar_type = time_var.to_index().calendar
    start_date_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    end_date_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
    
    # Adjust day to be 30 ( to avoid error if it's the 31st day in a 360_day calendar)
    start_date_cftime = cftime.datetime(start_date_dt.year, start_date_dt.month, min(start_date_dt.day, 30), calendar=calendar_type)
    end_date_cftime = cftime.datetime(end_date_dt.year, end_date_dt.month, min(end_date_dt.day, 30), calendar=calendar_type)
  
    # Check the selcted dates are within the range of model data
    check_datarange(time_var,start_date_cftime, end_date_cftime)
    
    # Put model into mascon space and calulate mass change of model data
    mass_change_mod_trim, mass_change_mod = transformToGeodetic(gsfc, gis_ds, start_date_cftime, end_date_cftime, rho_ice, rho_water, polar_stereographic)
    
    # Calculate mass change of model and observation data
    mass_change_delta = mass_change_mod_trim-mass_change_obs
    
    # Write result to nc file
    output_filename = os.path.splitext(os.path.basename(nc_filename))[0] + '_mascon_comp'
    output_netcdf_filename = output_netcdf_filepath + output_filename + '.nc'
    write_to_netcdf(mass_change_obs, mass_change_delta, mass_change_mod_trim, gsfc, I_, start_date_cftime, end_date_cftime, output_netcdf_filename)
